# DSPy as Orchestrator  vs  2-Agent Direct
## Head-to-head comparison — same 10 procurement emails

| | Pipeline A | Pipeline B |
|---|---|---|
| **Name** | DSPy Orchestrator | 2-Agent Direct |
| **Agents** | 1 SDK agent (enricher) | 2 SDK agents |
| **DSPy role** | **Owns the full handoff chain** | Not used |
| **Flow** | DSPy.forward → Predict(Extract) → Predict(Audit) → Agent(Enrich) | Agent1(Extract+Audit) → Agent2(Enrich+Validate) |

> **Key question:** Does giving DSPy full orchestration control produce better records than a clean 2-agent chain?


## 1 · Installation & Imports

In [2]:
!pip install openai-agents dspy -q

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [ ]:
import json, asyncio, dspy
from openai import AsyncAzureOpenAI
from agents import Agent, Runner, ModelSettings, set_default_openai_client, set_default_openai_api, set_tracing_disabled
import nest_asyncio
nest_asyncio.apply()
set_tracing_disabled(True)


## 2 · Credentials

In [ ]:
AZURE_API_KEY     = ""
AZURE_ENDPOINT    = ""
AZURE_API_VERSION = ""
AZURE_DEPLOYMENT  = ""

azure_client = AsyncAzureOpenAI(
    api_key=AZURE_API_KEY,
    azure_endpoint=AZURE_ENDPOINT,
    api_version=AZURE_API_VERSION,
)
set_default_openai_client(azure_client)
set_default_openai_api("chat_completions")

lm = dspy.LM(
    model       = f"azure/{AZURE_DEPLOYMENT}",
    api_key     = AZURE_API_KEY,
    api_base    = AZURE_ENDPOINT,
    api_version = AZURE_API_VERSION,
    max_tokens  = 2500,
    temperature = 0.0,
)
dspy.configure(lm=lm)
print("Azure + DSPy configured.")


## 3 · Load Emails

In [ ]:
with open("/content/bidding_emails.json", encoding="utf-8") as f:
    emails = json.load(f)
print(f"Loaded {len(emails)} emails.")
for k, v in emails.items():
    print(f"  {k} -> {v[:80].strip()}...")


---
## Pipeline A — DSPy as Orchestrator

```
Email Text
    │
    ▼  [Handoff 1 — DSPy Predict]
ExtractionSignature  →  extracted_json
    │
    ▼  [Handoff 2 — DSPy Predict]
AuditSignature       →  audited_json
    │
    ▼  [Handoff 3 — SDK Agent]
EnrichmentAgent      →  final_record
```

**DSPy owns every handoff.** The `ProcurementPipeline.forward()` method is the
single entry point — it decides when each step runs and what it receives.
The SDK agent is called *inside* DSPy's execution context, not the other way around.


In [ ]:
# ── DSPy Signatures ──────────────────────────────────────────────────────────

class ExtractionSignature(dspy.Signature):
    """
    You are a forensic document extractor. Read the procurement email and output a single JSON.

    CORE RULE: extracted_values contains ONLY facts the email explicitly states.
               field_inventory lists ALL fields this domain/record_type requires,
               with stated values copied in and "" for every missing required field.
               Never invent or assume any value.

    Output exactly 5 top-level keys:
      document_meta  — sender, recipient, date, title_or_subject, document_type
      context        — domain, record_type, summary (1-2 sentences), tone, intent
      extracted_values — every concrete fact explicitly stated (exact wording, numbers, dates)
      field_inventory  — complete field checklist for this domain/record_type:
                         identifiers, subject_item, scope_quantity, financial,
                         timeline, parties, location, requirements, contacts,
                         plus domain-specific fields derived from domain+record_type.
      open_items     — array of ambiguities, contradictions, or items needing clarification

    Raw JSON only. No markdown fences. No prose.
    """
    email_text    : str = dspy.InputField(desc="Raw procurement email text")
    extracted_json: str = dspy.OutputField(
        desc="JSON with keys: document_meta, context, extracted_values, field_inventory, open_items"
    )


class AuditSignature(dspy.Signature):
    """
    You are a JSON structure auditor. Structural work only — you do NOT fill values.

    Given extracted procurement JSON, do exactly three things:

    A. CLOSE GAPS — add any missing domain-standard fields to field_inventory as "".
       Ask: what would a senior reviewer immediately notice is absent?
       Common gaps: validity_period, payment_terms, force_majeure_clause,
       dispute_resolution, governing_law, evaluation_criteria, bid_validity_days.
       Do NOT modify or remove any existing field_inventory entries.

    B. QUARANTINE — move problematic field_inventory entries to review_section:
       TYPE 1 DUPLICATE  : { "value": v, "instruction": "DUPLICATE of <key> — merge into <key> and drop" }
       TYPE 2 IRRELEVANT : { "value": v, "instruction": "IRRELEVANT to <domain> <record_type> — drop" }
       TYPE 3 AMBIGUOUS  : { "value": v, "instruction": "AMBIGUOUS — rename to '<suggested_name>'" }
       TYPE 4 MISPLACED  : { "value": v, "instruction": "MISPLACED — move to <section>" }

    C. WRITE audit_summary:
       gaps_added          — array of key names added in step A
       quarantined         — array of key names moved in step B
       completeness_before — integer 0-100
       completeness_after  — integer 0-100
       next_agent_briefing — string with sub-sections:
         FILL: fill all "" values using domain knowledge; derive computed fields;
               use __NEEDS_INPUT__ only as last resort.
         RESOLVE: for each review_section key, execute its typed instruction.
         VALIDATE: run completeness, consistency, and risk checks.
         OUTPUT: return document_meta, context, extracted_values, field_inventory,
                 open_items, resolution_log, validation. Remove review_section, audit_summary.

    Preserve ALL existing keys. Only add or move — never silently delete.
    Raw JSON only. No markdown fences.
    """
    input_json   : str = dspy.InputField(
        desc="Extracted JSON from ExtractionSignature"
    )
    audited_json : str = dspy.OutputField(
        desc=(
            "Modified JSON with: field_inventory enriched with domain-standard fields as '', "
            "review_section with typed instructions, audit_summary with next_agent_briefing"
        )
    )


In [ ]:
# ── Enrichment Agent (SDK) — called by DSPy orchestrator as Handoff 3 ────────

ENRICHER_INSTRUCTIONS = """
You are a SENIOR DATA ENRICHMENT AND DECISION AGENT.
You receive a JSON that has been extracted and structurally audited from a procurement email.
Complete it, resolve all open questions, and return the final clean record.

READ FIRST:
  1. Read audit_summary.next_agent_briefing — it has your exact instructions.
  2. Note audit_summary.gaps_added — newly added fields needing values.
  3. Note audit_summary.quarantined — fields in review_section awaiting decisions.
  4. Do NOT change document_meta, context, or extracted_values — they are finalized.

TASK 1 — FILL field_inventory (highest priority):
  For every field where value is "":
    STEP 1 — DERIVE: compute from other filled fields (qty × price = total, date math, etc.)
    STEP 2 — DOMAIN KNOWLEDGE: standard/typical value for this field in this domain.
              Give specific values, state the basis (e.g. "Net 30 (industry standard)").
    STEP 3 — LAST RESORT: "__NEEDS_INPUT__" only if genuinely unknowable.

TASK 2 — RESOLVE review_section:
  DUPLICATE  → merge into canonical field, drop duplicate
  IRRELEVANT → drop (keep only if clearly applicable despite flag)
  AMBIGUOUS  → apply suggested rename and fill; drop if unresolvable
  MISPLACED  → move to the correct section
  Record every decision in resolution_log:
    { "<key>": { "action": "merged|dropped|renamed|moved", "destination": "...", "reason": "..." } }

TASK 3 — VALIDATE:
  Add a validation section:
    completeness   — "complete" | "mostly_complete" | "incomplete"
    missing_inputs — fields still __NEEDS_INPUT__
    inconsistencies — contradictions or mismatches between fields
    risk_flags     — concerns a reviewer would escalate
    summary        — one sentence on overall record quality and readiness

STRICT OUTPUT RULES:
  Return EXACTLY: document_meta, context, extracted_values, field_inventory,
                  open_items, resolution_log, validation
  REMOVE: review_section, audit_summary
  Every field_inventory value: filled or "__NEEDS_INPUT__" — no "" remaining
  Raw JSON only. No markdown fences.
"""

enricher_agent = Agent(
    name           = "EnrichmentAgent",
    model          = AZURE_DEPLOYMENT,
    model_settings = ModelSettings(max_tokens=4000, temperature=0.0),
    instructions   = ENRICHER_INSTRUCTIONS,
)
print("Enrichment agent ready.")


In [ ]:
# ── DSPy Orchestrator Module ─────────────────────────────────────────────────

class ProcurementPipeline(dspy.Module):
    """
    DSPy owns the full handoff chain:
      forward(email) → Predict(Extraction) → Predict(Audit) → Runner(EnrichmentAgent)

    All three handoffs happen inside DSPy's execution context.
    The entire pipeline is a single optimisable DSPy module.
    """

    def __init__(self, enricher_agent):
        super().__init__()
        self.extract  = dspy.Predict(ExtractionSignature)
        self.audit    = dspy.Predict(AuditSignature)
        self._enricher = enricher_agent

    async def forward(self, email_key: str, email_text: str) -> dict:
        sep = "=" * 56
        print(f"\n{sep}\n  [DSPy Orchestrator] {email_key}\n{sep}")

        # ── Handoff 1: raw email → DSPy Extraction ────────────────
        print("  [Handoff 1]  Email text  ──►  DSPy Extractor")
        step1        = self.extract(email_text=email_text)
        extracted    = step1.extracted_json
        print(f"               Done — {len(extracted)} chars")

        # ── Handoff 2: extracted JSON → DSPy Audit ────────────────
        print("  [Handoff 2]  Extracted JSON  ──►  DSPy Auditor")
        step2    = self.audit(input_json=extracted)
        audited  = step2.audited_json
        print(f"               Done — {len(audited)} chars")

        # ── Handoff 3: audited JSON → SDK Enrichment Agent ────────
        print("  [Handoff 3]  Audited JSON  ──►  SDK Enrichment Agent")
        result      = await Runner.run(self._enricher, input=audited)
        raw_final   = result.final_output
        print(f"               Done — {len(raw_final)} chars")

        # ── Parse ──────────────────────────────────────────────────
        try:
            clean  = raw_final.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
            parsed = json.loads(clean)
        except json.JSONDecodeError as e:
            print(f"  [WARNING] JSON parse failed: {e}")
            parsed = {"raw_output": raw_final, "parse_error": str(e)}

        return {
            "email_key"   : email_key,
            "pipeline"    : "A_dspy_orchestrator",
            "step1_chars" : len(extracted),
            "step2_chars" : len(audited),
            "step3_chars" : len(raw_final),
            "final"       : parsed,
        }


pipeline_a = ProcurementPipeline(enricher_agent)
print("Pipeline A (DSPy Orchestrator) ready.")


In [ ]:
# ── Run Pipeline A on all emails ─────────────────────────────────────────────

async def run_pipeline_a(emails: dict) -> list:
    results = []
    for key, body in emails.items():
        result = await pipeline_a.forward(email_key=key, email_text=body)
        results.append(result)
        print(f"\n[FINAL — {key}]")
        print(json.dumps(result["final"], indent=2, ensure_ascii=False))
    return results

results_a = await run_pipeline_a(emails)
print(f"\nPipeline A complete — {len(results_a)} emails processed.")


---
## Pipeline B — 2-Agent Direct (No DSPy)

```
Email Text
    │
    ▼  [Agent 1 — Extractor + Auditor combined]
  Structured JSON with field inventory, gaps marked, audit notes
    │
    ▼  [Direct handoff]
  [Agent 2 — Enricher + Validator combined]
    │
    ▼
  Final validated record
```

**No DSPy at all.** Agent 1 handles both extraction and structural auditing in one pass.
Agent 2 handles both enrichment and validation. Direct Agent1 → Agent2 handoff via Python.


In [ ]:
# ── Agent 1 — Extractor + Auditor (combined, single-pass) ────────────────────

AGENT1_INSTRUCTIONS = """
You are a COMBINED FORENSIC EXTRACTOR AND STRUCTURAL AUDITOR for procurement records.

You work in two phases in a single pass:

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
PHASE 1 — EXTRACTION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Read the email carefully. Capture every concrete fact the document explicitly states:
names, numbers, dates, amounts, specs, locations, contacts, terms, preferences.
Use the document's exact wording. Do NOT paraphrase or infer.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
PHASE 2 — STRUCTURAL AUDIT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. Classify the document: domain, record_type, tone, intent.
2. Derive the full required field set for this domain + record_type.
3. Build field_inventory: copy stated values in; set "" for required but unstated fields.
4. Identify and quarantine: duplicates, irrelevant fields, ambiguous keys.
5. Note any open items (ambiguities, contradictions, missing clarifications).

OUTPUT — exactly 6 top-level keys (raw JSON, no markdown fences):

  document_meta   — sender, recipient, date, title_or_subject, document_type
  context         — domain, record_type, summary (1-2 sentences), tone, intent
  extracted_values — ONLY facts explicitly stated in the email
  field_inventory — complete checklist for this domain/record_type:
                    - stated values copied from extracted_values
                    - "" for required fields not provided
                    - snake_case keys grouped into logical categories
                    - cover: identifiers, subject_item, scope_quantity, financial,
                      timeline, parties, location, requirements, contacts,
                      plus domain-specific fields
  open_items      — array of strings: ambiguities, contradictions, items needing clarification
  audit_notes     — array of strings: fields added beyond what the email states, with reason

STRICT RULES:
  • Never invent values in extracted_values
  • Never leave a standard required field out of field_inventory
  • Raw JSON only. No markdown fences. Valid JSON.
"""

agent1_direct = Agent(
    name           = "ExtractorAuditor",
    model          = AZURE_DEPLOYMENT,
    model_settings = ModelSettings(max_tokens=3000, temperature=0.0),
    instructions   = AGENT1_INSTRUCTIONS,
)


# ── Agent 2 — Enricher + Validator (combined) ─────────────────────────────────

AGENT2_INSTRUCTIONS = """
You are a SENIOR DATA ENRICHMENT AND VALIDATION AGENT.

You receive a structured JSON that has been extracted and audited from a procurement email.
Your job: complete it, resolve open items, and return the final validated record.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
TASK 1 — FILL field_inventory (highest priority)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
For every field where value is "" (empty string):

  STEP 1 — DERIVE: Can you compute it from other filled fields?
           (e.g. total_value = quantity × unit_price, end_date = start + duration)
  STEP 2 — DOMAIN KNOWLEDGE: What is the standard/typical value for this field?
           Give specific values, not vague placeholders.
           State the basis: e.g. "Net 30 (industry standard)", "UN38.3 (EV battery standard)"
  STEP 3 — LAST RESORT: "__NEEDS_INPUT__" only if genuinely unknowable without more input.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
TASK 2 — RESOLVE open_items and audit_notes
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Address each open_item. If resolvable using domain knowledge, apply the resolution.
Record every decision in resolution_log:
  { "<item>": { "action": "resolved|escalated|clarified", "outcome": "...", "reason": "..." } }

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
TASK 3 — VALIDATE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Add a validation section:
  completeness    — "complete" | "mostly_complete" | "incomplete"
  missing_inputs  — array of field_inventory keys still set to "__NEEDS_INPUT__"
  inconsistencies — array of contradictions or conflicts found between fields
  risk_flags      — array of concerns a reviewer would escalate (be specific)
  summary         — one sentence assessing overall record quality and readiness

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
STRICT OUTPUT RULES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Return EXACTLY these top-level keys:
  document_meta, context, extracted_values, field_inventory,
  open_items, resolution_log, validation
Every field_inventory value: filled or "__NEEDS_INPUT__" — no "" remaining.
Raw JSON only. No markdown fences. Valid JSON.
"""

agent2_direct = Agent(
    name           = "EnrichmentValidator",
    model          = AZURE_DEPLOYMENT,
    model_settings = ModelSettings(max_tokens=4000, temperature=0.0),
    instructions   = AGENT2_INSTRUCTIONS,
)
print("Pipeline B agents ready.")


In [ ]:
# ── Run Pipeline B on all emails ─────────────────────────────────────────────

async def run_pipeline_b(emails: dict) -> list:
    results = []
    for key, body in emails.items():
        sep = "=" * 56
        print(f"\n{sep}\n  [2-Agent Direct] {key}\n{sep}")

        # ── Agent 1: Extract + Audit ───────────────────────────────
        print("  [Agent 1]   Extracting and auditing...")
        r1 = await Runner.run(agent1_direct, input=body)
        print(f"              Done — {len(r1.final_output)} chars")

        # ── Direct handoff: Agent1 → Agent2 ───────────────────────
        print("  [Handoff]   Agent 1 output  ──►  Agent 2")
        print("  [Agent 2]   Enriching and validating...")
        r2 = await Runner.run(agent2_direct, input=r1.final_output)
        print(f"              Done — {len(r2.final_output)} chars")

        # ── Parse ──────────────────────────────────────────────────
        try:
            clean  = r2.final_output.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
            parsed = json.loads(clean)
        except json.JSONDecodeError as e:
            print(f"  [WARNING] JSON parse failed: {e}")
            parsed = {"raw_output": r2.final_output, "parse_error": str(e)}

        result = {
            "email_key"   : key,
            "pipeline"    : "B_2agent_direct",
            "agent1_chars": len(r1.final_output),
            "agent2_chars": len(r2.final_output),
            "final"       : parsed,
        }
        results.append(result)
        print(f"\n[FINAL — {key}]")
        print(json.dumps(parsed, indent=2, ensure_ascii=False))

    return results

results_b = await run_pipeline_b(emails)
print(f"\nPipeline B complete — {len(results_b)} emails processed.")


---
## Comparison — Pipeline A vs Pipeline B

Metrics per email:
- **Parse success** — did the final output parse as valid JSON?
- **Fields filled** — how many field_inventory values are non-empty and not `__NEEDS_INPUT__`?
- **Missing inputs** — how many fields still need human input?
- **Risk flags** — how many real risks were surfaced?
- **Completeness** — the pipeline's own self-assessment


In [ ]:
def analyse(result: dict) -> dict:
    """Extract key metrics from a pipeline result."""
    final = result["final"]

    if "parse_error" in final:
        return {"email_key": result["email_key"], "pipeline": result["pipeline"],
                "parse_ok": False, "fields_filled": 0, "missing_inputs": 0,
                "risk_flags": 0, "completeness": "parse_error"}

    fi = final.get("field_inventory", {})

    def count_values(obj, filled=0, missing=0):
        """Recursively count filled vs __NEEDS_INPUT__ leaf values."""
        if isinstance(obj, dict):
            for v in obj.values():
                filled, missing = count_values(v, filled, missing)
        elif isinstance(obj, list):
            for v in obj:
                filled, missing = count_values(v, filled, missing)
        elif isinstance(obj, str):
            if obj == "__NEEDS_INPUT__":
                missing += 1
            elif obj not in ("", None):
                filled += 1
        return filled, missing

    filled, missing = count_values(fi)
    val     = final.get("validation", {})
    rf      = val.get("risk_flags", [])
    compl   = val.get("completeness", "—")

    return {
        "email_key"    : result["email_key"],
        "pipeline"     : result["pipeline"],
        "parse_ok"     : True,
        "fields_filled": filled,
        "missing_inputs": missing,
        "risk_flags"   : len(rf) if isinstance(rf, list) else 0,
        "completeness" : compl,
        "output_chars" : result.get("step3_chars") or result.get("agent2_chars", 0),
    }

metrics_a = [analyse(r) for r in results_a]
metrics_b = [analyse(r) for r in results_b]

# ── Print comparison table ────────────────────────────────────────────────────
print("\n" + "=" * 100)
print(f"{'Email':<10} {'Parse':>6}  {'Filled':>7} {'Miss':>6} {'Flags':>6} {'Completeness':<18}  |  "
      f"{'Parse':>6}  {'Filled':>7} {'Miss':>6} {'Flags':>6} {'Completeness':<18}")
print(f"{'':10} {'A':>6}  {'A':>7} {'A':>6} {'A':>6} {'A':<18}  |  "
      f"{'B':>6}  {'B':>7} {'B':>6} {'B':>6} {'B':<18}")
print("-" * 100)

for ma, mb in zip(metrics_a, metrics_b):
    def fmt(m):
        p  = "OK" if m["parse_ok"] else "FAIL"
        return (f"{p:>6}  {m['fields_filled']:>7} {m['missing_inputs']:>6} "
                f"{m['risk_flags']:>6} {m['completeness']:<18}")
    print(f"{ma['email_key']:<10} {fmt(ma)}  |  {fmt(mb)}")

print("=" * 100)

# ── Totals ────────────────────────────────────────────────────────────────────
def totals(metrics):
    ok   = sum(1 for m in metrics if m["parse_ok"])
    fill = sum(m["fields_filled"]  for m in metrics if m["parse_ok"])
    miss = sum(m["missing_inputs"] for m in metrics if m["parse_ok"])
    risk = sum(m["risk_flags"]     for m in metrics if m["parse_ok"])
    return ok, fill, miss, risk

ok_a, fill_a, miss_a, risk_a = totals(metrics_a)
ok_b, fill_b, miss_b, risk_b = totals(metrics_b)

print(f"\n{'TOTALS':<10} {'':>6}  {fill_a:>7} {miss_a:>6} {risk_a:>6} {'':18}  |  "
      f"{'':>6}  {fill_b:>7} {miss_b:>6} {risk_b:>6}")
print(f"\nPipeline A — {ok_a}/10 parsed  |  {fill_a} fields filled  |  {miss_a} missing  |  {risk_a} risk flags")
print(f"Pipeline B — {ok_b}/10 parsed  |  {fill_b} fields filled  |  {miss_b} missing  |  {risk_b} risk flags")


---
## Architecture Trade-offs

| Dimension | Pipeline A — DSPy Orchestrator | Pipeline B — 2-Agent Direct |
|---|---|---|
| **Who controls flow** | `ProcurementPipeline.forward()` — DSPy | Your Python `run_pipeline_b()` |
| **Optimisable?** | Yes — `BootstrapFewShot` can tune Extract + Audit signatures jointly | No — agent instructions are static strings |
| **Handoff transparency** | DSPy traces each step; accessible via `dspy.inspect_history()` | Manual — you see Agent1/Agent2 outputs directly |
| **Token usage** | 3 LLM calls (2 DSPy + 1 agent) | 2 LLM calls (agent1 + agent2) |
| **Agent 1 burden** | Light — extraction only | Heavy — extraction + auditing combined |
| **Failure isolation** | Any step can be individually re-run or replaced | If Agent1 misses a structural issue, Agent2 must catch it |
| **Unseen domains** | DSPy Predict derives schema dynamically per run | Agent1 instructions are static — novel domains may be under-specified |
| **Best for** | Pipelines you want to improve over time via DSPy optimization | Simple, fast deployments where you control prompts manually |


In [ ]:
# Inspect DSPy execution history (Pipeline A only)
print("── DSPy execution history (last 2 calls) ──")
dspy.inspect_history(n=2)
